In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Вказання параметрів

<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці було розроблено з використанням таких вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ти можеш використовувати параметри для налаштування примітивів Estimator та Sampler. Цей розділ зосереджується на тому, як вказувати параметри примітивів Qiskit Runtime. Хоча інтерфейс методу `run()` примітивів є спільним для всіх реалізацій, їхні параметри відрізняються. Зверни увагу на відповідні довідники API для отримання інформації про параметри [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) та [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

Примітки щодо вказання параметрів у примітивах:

- `SamplerV2` та `EstimatorV2` мають окремі класи параметрів. Ти можеш переглядати доступні параметри та оновлювати їхні значення під час або після ініціалізації примітива.
- Використовуй метод `update()` для застосування змін до атрибута `options`.
- Якщо ти не вказуєш значення для параметра, йому присвоюється спеціальне значення `Unset` і використовуються серверні значення за замовчуванням.
- Атрибут `options` є типом Python `dataclass`. Ти можеш використовувати вбудований метод `asdict` для перетворення його в словник.

<span id="pass-options"></span>
## Встановлення параметрів примітива
Ти можеш встановлювати параметри під час ініціалізації примітива, після ініціалізації або в методі `run()`. Перегляньте розділ [правила пріоритетності](runtime-options-overview#options-precedence), щоб зрозуміти, що відбувається, коли той самий параметр вказано в кількох місцях.

### Ініціалізація примітива
Під час ініціалізації примітива ти можеш передати екземпляр класу параметрів або словник, після чого примітив робить копію цих параметрів. Таким чином, зміна оригінального словника або екземпляра параметрів не впливає на параметри, якими володіє примітив.

#### Клас параметрів
При створенні екземпляра класу `EstimatorV2` або `SamplerV2` можна передати екземпляр класу параметрів. Ці параметри будуть застосовані при виконанні розрахунку за допомогою `run()`. Вказуй параметри в такому форматі: `options.option.sub-option.sub-sub-option = choice`. Наприклад: `options.dynamical_decoupling.enable = True`

Приклад:

`SamplerV2` та `EstimatorV2` мають окремі класи параметрів ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) та [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Словник
Ти можеш вказати параметри у вигляді словника під час ініціалізації примітива.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Оновлення параметрів після ініціалізації
Ти можеш вказати параметри у форматі `primitive.options.option.sub-option.sub-sub-option = choice`, щоб скористатися автодоповненням, або використати метод `update()` для масового оновлення.

Класи параметрів `SamplerV2` та `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) та [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) не потрібно інстанціювати, якщо ти встановлюєш параметри після ініціалізації примітива.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Метод Run()
Єдині значення, які можна передати в `run()`, — це ті, що визначені в інтерфейсі. Тобто `shots` для Sampler та `precision` для Estimator. Це перевизначає будь-яке значення, встановлене для `default_shots` або `default_precision` для поточного запуску.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Особливі випадки
#### Рівень стійкості (лише для Estimator)
Рівень стійкості насправді не є параметром, що безпосередньо впливає на запит до примітива, але задає базовий набір ретельно відібраних параметрів для побудови конфігурації. Загалом, рівень 0 вимикає всі методи пом'якшення помилок, рівень 1 вмикає параметри для пом'якшення помилок вимірювання, а рівень 2 вмикає параметри для пом'якшення помилок вентилів і вимірювань.

Будь-які параметри, які ти вказуєш додатково до рівня стійкості, застосовуються поверх базового набору параметрів, визначеного рівнем стійкості. Тому в принципі ти міг би встановити рівень стійкості на 1, але потім вимкнути пом'якшення помилок вимірювання, хоча це не рекомендується.

У наступному прикладі встановлення рівня стійкості на 0 спочатку вимикає `zne_mitigation`, але `estimator.options.resilience.zne_mitigation = True` перевизначає відповідне налаштування з `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Кількість вимірювань (лише для Sampler)
Метод `SamplerV2.run` приймає два аргументи: список PUB, кожен з яких може вказувати специфічне для PUB значення shots, та ключовий аргумент shots. Ці значення shots є частиною інтерфейсу виконання Sampler і незалежні від параметрів Runtime Sampler. Вони мають пріоритет над будь-якими значеннями, вказаними як параметри, щоб відповідати абстракції Sampler.

Однак якщо `shots` не вказано ні в якому PUB, ні в ключовому аргументі run (або якщо всі вони є `None`), то використовується значення shots з параметрів, насамперед `default_shots`.

Підсумовуючи, ось порядок пріоритетності для вказання shots у Sampler для будь-якого конкретного PUB:

1. Якщо PUB вказує shots, використовується це значення.
2. Якщо в `run` вказано ключовий аргумент `shots`, використовується це значення.
3. Якщо `num_randomizations` та `shots_per_randomization` вказані як параметри `twirling`, shots є добутком цих значень.
3. Якщо вказано `sampler.options.default_shots`, використовується це значення.

Таким чином, якщо shots вказано в усіх можливих місцях, використовується те, що має найвищий пріоритет (shots, вказане в PUB).

#### Точність (лише для Estimator)
Точність є аналогом shots, описаного в попередньому розділі, за винятком того, що параметри Estimator містять як `default_shots`, так і `default_precision`. Крім того, оскільки gate-twirling увімкнено за замовчуванням, добуток `num_randomizations` та `shots_per_randomization` має пріоритет над цими двома параметрами.

Зокрема, для будь-якого конкретного PUB Estimator:

1. Якщо PUB вказує точність, використовується це значення.
2. Якщо в `run` вказано ключовий аргумент precision, використовується це значення.
2. Якщо `num_randomizations` та `shots_per_randomization` вказані як [параметри `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (увімкнено за замовчуванням), використовується їхній добуток для керування обсягом даних.
3. Якщо вказано `estimator.options.default_shots`, використовується це значення для керування обсягом даних.
4. Якщо вказано `estimator.options.default_precision`, використовується це значення.

Наприклад, якщо точність вказана у всіх чотирьох місцях, використовується те, що має найвищий пріоритет (точність, вказана в PUB).

> **Note:** Точність обернено масштабується з використанням. Тобто, що нижча точність, то більше часу QPU потрібно для виконання.

## Часто використовувані параметри
Існує багато доступних параметрів, але найчастіше використовуються такі:

<span id="shots"></span>
### Кількість вимірювань
Для деяких алгоритмів встановлення конкретної кількості вимірювань є ключовою частиною їхніх процедур. Кількість вимірювань (або точність) можна вказати в кількох місцях. Вони мають такий пріоритет:

Для будь-якого PUB Sampler:

1. Ціле значення shots, що міститься в PUB
2. Значення `run(...,shots=val)`
3. Значення `options.default_shots`

Для будь-якого PUB Estimator:

1. Значення precision з плаваючою точкою, що міститься в PUB
2. Значення `run(...,precision=val)`
3. Значення `options.default_shots`
4. Значення `options.default_precision`

Приклад:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Максимальний час виконання
Максимальний час виконання (`max_execution_time`) обмежує тривалість роботи завдання. Якщо завдання перевищує цей ліміт часу, воно примусово скасовується. Це значення застосовується до окремих завдань незалежно від того, чи запускаються вони в режимі завдання, сесії або пакетному режимі.

Значення встановлюється в секундах на основі квантового часу (не часу за годинником), тобто часу, протягом якого QPU присвячений обробці твого завдання. Воно ігнорується при використанні режиму локального тестування, оскільки цей режим не використовує квантовий час.